In [1]:
!pip install -q spacy networkx plotly pandas requests
!python -m spacy download en_core_web_sm -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 109.1 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [2]:
import re
import requests
import pandas as pd
import spacy
import networkx as nx
import plotly.graph_objects as go

from IPython.display import display

nlp = spacy.load("en_core_web_sm")

WIKI_API = "https://en.wikipedia.org/w/api.php"

HEADERS = {
    "User-Agent": "WikipediaKnowledgeGraphProject/1.0 (educational use)"
}

print("Libraries loaded successfully!")

Libraries loaded successfully!


In [3]:
topic = "Artificial Intelligence"
number_of_articles = 5

def search_wikipedia(topic, limit=5):
    params = {
        "action": "query",
        "list": "search",
        "srsearch": topic,
        "srlimit": limit,
        "format": "json"
    }

    response = requests.get(
        WIKI_API,
        params=params,
        headers=HEADERS,
        timeout=30
    )

    response.raise_for_status()

    return response.json()["query"]["search"]


def fetch_article(title):
    params = {
        "action": "query",
        "prop": "extracts",
        "explaintext": 1,
        "titles": title,
        "format": "json",
        "redirects": 1
    }

    response = requests.get(
        WIKI_API,
        params=params,
        headers=HEADERS,
        timeout=30
    )

    response.raise_for_status()

    pages = response.json()["query"]["pages"]
    page = next(iter(pages.values()))

    return page.get("title", title), page.get("extract", "")


search_results = search_wikipedia(
    topic,
    number_of_articles
)

print("Topic:", topic)
print("Articles found:", len(search_results))

for item in search_results:
    print(item["title"])

Topic: Artificial Intelligence
Articles found: 5
Artificial intelligence
Artificial general intelligence
A.I. Artificial Intelligence
History of artificial intelligence
Hallucination (artificial intelligence)


In [4]:
def extract_entities_and_relations(text):

    doc = nlp(text[:100000])

    entities = set()
    triples = []

    excluded_labels = {
        "CARDINAL", "ORDINAL", "DATE", "TIME",
        "MONEY", "PERCENT", "QUANTITY"
    }

    for ent in doc.ents:
        name = ent.text.strip(" ,.;:")

        if len(name) > 1 and ent.label_ not in excluded_labels:
            entities.add(name)

    for sentence in doc.sents:

        sentence_entities = [
            ent for ent in sentence.ents
            if ent.label_ not in excluded_labels
        ]

        for left, right in zip(
            sentence_entities,
            sentence_entities[1:]
        ):

            subject = left.text.strip()
            obj = right.text.strip()

            if subject.casefold() == obj.casefold():
                continue

            between = sentence.text[
                left.end_char - sentence.start_char:
                right.start_char - sentence.start_char
            ]

            words = re.findall(
                r"[A-Za-z'-]+",
                between.lower()
            )

            stop_words = {
                "the", "a", "an", "of", "in", "on",
                "at", "to", "and", "or", "for", "with",
                "by", "from", "as", "is", "was", "are",
                "were", "that", "which"
            }

            words = [
                word for word in words
                if word not in stop_words
            ]

            relation = (
                " ".join(words[-5:])
                if words else "mentioned with"
            )

            triples.append({
                "Subject": subject,
                "Relation": relation,
                "Object": obj,
                "Evidence": sentence.text.strip()
            })

    # Remove duplicate triples
    unique_triples = []
    seen = set()

    for triple in triples:

        key = (
            triple["Subject"].casefold(),
            triple["Relation"].casefold(),
            triple["Object"].casefold()
        )

        if key not in seen:
            seen.add(key)
            unique_triples.append(triple)

    return entities, unique_triples

In [5]:
all_triples = []
article_details = []

for result in search_results:

    title, article_text = fetch_article(
        result["title"]
    )

    if not article_text.strip():
        continue

    entities, triples = extract_entities_and_relations(
        article_text
    )

    all_triples.extend(triples)

    article_details.append({
        "Article": title,
        "Wikipedia URL":
            "https://en.wikipedia.org/wiki/"
            + requests.utils.quote(title.replace(" ", "_")),
        "Entities found": len(entities),
        "Relations found": len(triples)
    })

# Remove duplicate triples across articles
unique_triples = []
seen = set()

for triple in all_triples:

    key = (
        triple["Subject"].casefold(),
        triple["Relation"].casefold(),
        triple["Object"].casefold()
    )

    if key not in seen:
        seen.add(key)
        unique_triples.append(triple)

triples_df = pd.DataFrame(unique_triples)

print("Articles processed:", len(article_details))
print("Total unique triples:", len(triples_df))

display(triples_df.head(20))

Articles processed: 5
Total unique triples: 1563


,Subject,Relation,Object,Evidence
0,OpenAI,mentioned with,Google DeepMind,"Some companies, such as OpenAI, Google DeepMin..."
1,Google DeepMind,mentioned with,Meta,"Some companies, such as OpenAI, Google DeepMin..."
2,British,linguist philosopher,Margaret Masterman,British linguist and philosopher Margaret Mast...
3,SAT,scholastic assessment test,GRE,"By 2023, these models were able to get human-l..."
4,GRE,mentioned with,Graduate Record Examination,"By 2023, these models were able to get human-l..."
5,Horn,underpins computation logic programming language,Prolog,"However, backward reasoning with Horn clauses,..."
6,Prolog,mentioned with,Turing,"However, backward reasoning with Horn clauses,..."
7,Markov,models,Kalman,Probabilistic algorithms can also be used for ...
8,AI,until mid- s,Kernel,K-nearest neighbour algorithm was the most wid...
9,Kernel,methods such support vector machine,SVM,K-nearest neighbour algorithm was the most wid...


In [6]:
G = nx.DiGraph()

for triple in unique_triples:

    subject = triple["Subject"]
    relation = triple["Relation"]
    obj = triple["Object"]

    G.add_node(subject)
    G.add_node(obj)

    if G.has_edge(subject, obj):

        existing = G[subject][obj]["relation"]

        if relation not in existing:
            G[subject][obj]["relation"] += "; " + relation

    else:
        G.add_edge(
            subject,
            obj,
            relation=relation
        )

print("Entities (nodes):", G.number_of_nodes())
print("Relations (edges):", G.number_of_edges())

Entities (nodes): 1286
Relations (edges): 1461


In [1]:
if G.number_of_nodes() == 0:

    print("No entities found. Try another topic.")

else:

    pos = nx.spring_layout(
        G,
        seed=42,
        k=0.9
    )

    edge_x = []
    edge_y = []

    for source, target in G.edges():

        x0, y0 = pos[source]
        x1, y1 = pos[target]

        edge_x.extend([x0, x1, None])
        edge_y.extend([y0, y1, None])

    edge_trace = go.Scatter(
        x=edge_x,
        y=edge_y,
        mode="lines",
        line=dict(width=1),
        hoverinfo="none"
    )

    node_x = []
    node_y = []
    node_labels = []
    node_hover = []

    for node in G.nodes():

        x, y = pos[node]

        node_x.append(x)
        node_y.append(y)
        node_labels.append(node)

        node_hover.append(
            f"Entity: {node}<br>"
            f"Connections: {G.degree(node)}"
        )

    node_trace = go.Scatter(
        x=node_x,
        y=node_y,
        mode="markers+text",
        text=node_labels,
        textposition="top center",
        hovertext=node_hover,
        hoverinfo="text",
        marker=dict(
            size=18,
            line=dict(width=1)
        )
    )

    fig = go.Figure(
        data=[edge_trace, node_trace]
    )

    fig.update_layout(
        title=f"Knowledge Graph: {topic}",
        height=700,
        showlegend=False,
        xaxis=dict(visible=False),
        yaxis=dict(visible=False)
    )

    fig.show()

NameError: name 'G' is not defined

In [8]:
articles_df = pd.DataFrame(article_details)

display(articles_df)

,Article,Wikipedia URL,Entities found,Relations found
0,Artificial intelligence,https://en.wikipedia.org/wiki/Artificial_intel...,348,307
1,Artificial general intelligence,https://en.wikipedia.org/wiki/Artificial_gener...,122,122
2,A.I. Artificial Intelligence,https://en.wikipedia.org/wiki/A.I._Artificial_...,243,339
3,History of artificial intelligence,https://en.wikipedia.org/wiki/History_of_artif...,750,652
4,Hallucination (artificial intelligence),https://en.wikipedia.org/wiki/Hallucination_%2...,196,166


In [9]:
triples_df.to_csv(
    "knowledge_graph_triples.csv",
    index=False
)

articles_df.to_csv(
    "wikipedia_source_articles.csv",
    index=False
)

print("CSV files created successfully!")

from google.colab import files

files.download("knowledge_graph_triples.csv")
files.download("wikipedia_source_articles.csv")

CSV files created successfully!


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>